# 🚀 SparseChron — 4D Interactive Viewer (Official Retrained Model)

Streamlined, zero-bloat demo runner. Pre-configured for the **Official Retrained Model** (PSNR 29.17 / SSIM 0.945 / LPIPS 0.083) with pristine, stable colors and zero artifacting.

### Requirements
- **Session options:** Accelerator = **GPU T4**, Internet = **ON**
- **Total setup time:** ~30 seconds (ZERO compilation, pre-cached CUDA kernels, Cloudflare tunnel)

Run the 3 cells below in order.

In [ ]:
# 1. Clone repo + install dependencies (ZERO COMPILE, ~35-40s)
import os, subprocess, sys

REPO = "/kaggle/working/SparseChron"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/rajhodedara/SparseChron.git", REPO], check=True)
else:
    print("Repo already cloned, skipping.")

%cd /kaggle/working/SparseChron

# Unpack precompiled CUDA kernels directly into PyTorch JIT cache (~2s)
print("Unpacking precompiled gsplat CUDA kernels...")
!mkdir -p /root/.cache/torch_extensions
!curl -sL https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/gsplat_cache.tar.gz | tar -xz -C /root/.cache/torch_extensions

# Install gsplat wheel and Python dependencies (~35s network download)
print("Installing dependencies...")
WHEEL = "https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/gsplat-1.5.3-py3-none-any.whl"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", WHEEL])
!pip install -q -r requirements.txt

# Pre-register compiled binary into sys.modules BEFORE backend ever imports
from torch.utils.cpp_extension import _get_build_directory, _import_module_from_library
build_dir = _get_build_directory("gsplat_cuda", verbose=False)
_C = _import_module_from_library("gsplat_cuda", build_dir, is_python_module=True)
sys.modules["gsplat.csrc"] = _C
sys.modules["gsplat_cuda"] = _C

import gsplat.cuda._backend as _b
_b._C = _C

print("\n🚀 Environment ready! gsplat loaded from precompiled cache. ZERO compile time.")


In [ ]:
# 2. Download 347k 4D Model + Ensure clean colors on main branch
import os, urllib.request

# Ensure on main branch with clean color fix (DC base colors, no rainbow artifacts)
!cd /kaggle/working/SparseChron && git checkout main && git pull origin main

os.makedirs("/kaggle/working/outputs_mutant", exist_ok=True)
CKPT_PATH = "/kaggle/working/outputs_mutant/checkpoint_august_30000.ckpt"

if not os.path.exists(CKPT_PATH):
    print("Downloading 4D model checkpoint (347k Gaussians, 237 MB)...")
    urllib.request.urlretrieve(
        "https://github.com/rajhodedara/SparseChron/releases/download/trained-checkpoints/checkpoint_august_30000.ckpt",
        CKPT_PATH
    )
    print(f"Downloaded model ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)!")
else:
    print("4D model checkpoint already cached.")

print("\n🚀 347k Model staged on main branch! Clean colors active.")


In [ ]:
# 3. Launch 4D Viewer (Viser) + Cloudflare Tunnel (NO PASSWORDS, Clean Colors)
import os, subprocess, time, socket, glob

# 1. Clean up old processes
!pkill -9 -f interactive_viewer; !pkill -9 -f cloudflared; !pkill -9 -f localtunnel
time.sleep(1)

# 2. Download Cloudflare binary if not present (~2s)
if not os.path.exists("/kaggle/working/cloudflared"):
    print("Downloading Cloudflare tunnel binary...")
    !curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /kaggle/working/cloudflared
    !chmod +x /kaggle/working/cloudflared

# 3. Launch 4D viewer with the 347k model on main branch
CKPT_PATH = "/kaggle/working/outputs_mutant/checkpoint_august_30000.ckpt"
print(f"Booting 4D viewer with model: {CKPT_PATH}")

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
subprocess.Popen(
    ["python", "/kaggle/working/SparseChron/scripts/interactive_viewer.py", "--checkpoint", CKPT_PATH, "--port", "8080"],
    env=env
)

# 4. Wait for Viser to bind to port 8080
print("Loading 347,000 Gaussians into GPU VRAM (waiting for port 8080)...")
for _ in range(40):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(("127.0.0.1", 8080)) == 0:
            print("\n✅ Viser 4D Viewer is LIVE on port 8080!")
            break
    time.sleep(1)

# 5. Launch Cloudflare Tunnel (prints a clean https://*.trycloudflare.com link)
print("Opening Cloudflare Tunnel (NO PASSWORDS NEEDED)...\n")
!/kaggle/working/cloudflared tunnel --url http://127.0.0.1:8080
